<a href="https://colab.research.google.com/github/kauazin05/CIAO_KKG_2026/blob/main/lab1_aula_10_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip install scikit-fuzzy pytest -q
!mkdir -p AULA_11/src

In [16]:
%%writefile AULA_11/src/model.py
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import skfuzzy as fuzzy
from skfuzzy import control as ctrl

def criar_sistema_fuzzy():
    impacto = ctrl.Antecedent(np.arange(0, 101, 1), 'impacto')
    explorabilidade = ctrl.Antecedent(np.arange(0, 11, 1), 'explorabilidade')
    urgencia = ctrl.Consequent(np.arange(0, 101, 1), 'urgencia')

    urgencia.defuzzify_method = 'centroid'

    impacto['baixo'] = fuzzy.trimf(impacto.universe, [0, 0, 40])
    impacto['medio'] = fuzzy.trimf(impacto.universe, [20, 50, 80])
    impacto['alto'] = fuzzy.trapmf(impacto.universe, [60, 80, 100, 100])

    explorabilidade['baixa'] = fuzzy.trimf(explorabilidade.universe, [0, 0, 4])
    explorabilidade['moderada'] = fuzzy.trimf(explorabilidade.universe, [2, 5, 8])
    explorabilidade['alta'] = fuzzy.trimf(explorabilidade.universe, [6, 10, 10])

    urgencia['baixa'] = fuzzy.trimf(urgencia.universe, [0, 0, 30])
    urgencia['media'] = fuzzy.trimf(urgencia.universe, [20, 50, 70])
    urgencia['alta'] = fuzzy.trimf(urgencia.universe, [60, 80, 90])
    urgencia['critica'] = fuzzy.trapmf(urgencia.universe, [80, 90, 100, 100])

    return impacto, explorabilidade, urgencia


def gerar_graficos_pertinencia():
    impacto, explorabilidade, urgencia = criar_sistema_fuzzy()
    Path("results").mkdir(exist_ok=True)
    fig, (ax0, ax1, ax2) = plt.subplots(nrows=3, figsize=(8, 8))
    impacto.view(ax=ax0)
    explorabilidade.view(ax=ax1)
    urgencia.view(ax=ax2)
    plt.tight_layout()
    plt.savefig('results/membership_functions.png')
    plt.close()
    print("Sucesso: Grafico 'results/membership_functions.png' gerado com exito!")


if __name__ == "__main__":
    gerar_graficos_pertinencia()

Writing AULA_11/src/model.py


In [21]:
%%writefile AULA_11/src/engine.py
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from skfuzzy import control as ctrl
from src.model import criar_sistema_fuzzy


def construir_motor_inferencia():
    impacto, explorabilidade, urgencia = criar_sistema_fuzzy()

    regra1 = ctrl.Rule(impacto['baixo'] & explorabilidade['baixa'], urgencia['baixa'])
    regra2 = ctrl.Rule(impacto['baixo'] & explorabilidade['moderada'], urgencia['baixa'])
    regra3 = ctrl.Rule(impacto['baixo'] & explorabilidade['alta'], urgencia['media'])
    regra4 = ctrl.Rule(impacto['medio'] & explorabilidade['baixa'], urgencia['media'])
    regra5 = ctrl.Rule(impacto['medio'] & explorabilidade['moderada'], urgencia['alta'])
    regra6 = ctrl.Rule(impacto['medio'] & explorabilidade['alta'], urgencia['alta'])
    regra7 = ctrl.Rule(impacto['alto'] & explorabilidade['baixa'], urgencia['critica'])
    regra8 = ctrl.Rule(impacto['alto'] & explorabilidade['moderada'], urgencia['critica'])
    regra9 = ctrl.Rule(impacto['alto'] & explorabilidade['alta'], urgencia['critica'])

    sistema_controle = ctrl.ControlSystem([
        regra1, regra2, regra3,
        regra4, regra5, regra6,
        regra7, regra8, regra9
    ])
    return ctrl.ControlSystemSimulation(sistema_controle)


def gerar_superficie_3d(simulador):
    Path("results").mkdir(exist_ok=True)
    x, y = np.meshgrid(np.linspace(0, 100, 20), np.linspace(0, 10, 20))
    z = np.zeros_like(x)

    for i in range(20):
        for j in range(20):
            simulador.input['impacto'] = x[i, j]
            simulador.input['explorabilidade'] = y[i, j]
            simulador.compute()
            z[i, j] = simulador.output['urgencia']

    fig = plt.figure(figsize=(9, 6))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(x, y, z, cmap='viridis', edgecolor='none', alpha=0.9)

Writing AULA_11/src/engine.py


FileNotFoundError: [Errno 2] No such file or directory: 'AULA_11/src/engine.py'

In [17]:
%%writefile AULA_11/test_fuzzy_engine.py
# ==============================================================================
# AULA 11 - SPRINT 3 (AC-3): SUITE DE TESTES AUTOMATIZADOS (PYTEST)
# Objetivo: Validacao de Borda, Monotonicidade e Desempenho do Motor Fuzzy
# ==============================================================================

import time
import pytest
import numpy as np
from src.engine import construir_motor_inferencia


@pytest.fixture
def simulador():
    """Fixture que instancia o motor de inferencia compilado para os testes."""
    return construir_motor_inferencia()


def executar_inferencia(simulador, impacto, explorabilidade):
    """Funcao auxiliar para executar inferencia com validacao de entrada."""
    if not (0 <= impacto <= 100):
        raise ValueError(f"Impacto fora da escala [0, 100]: {impacto}")
    if not (0 <= explorabilidade <= 10):
        raise ValueError(f"Explorabilidade fora da escala [0, 10]: {explorabilidade}")

    simulador.input['impacto'] = impacto
    simulador.input['explorabilidade'] = explorabilidade
    simulador.compute()
    return simulador.output['urgencia']


# ------------------------------------------------------------------------------
# 1. TESTES DE LIMITES E INTERVALO DE SAIDA
# ------------------------------------------------------------------------------
def test_saida_dentro_do_universo(simulador):
    """Garante que a saida esteja sempre no intervalo [0, 100]%."""
    casos = [(0, 0), (50, 5), (100, 10), (30, 8), (80, 2)]
    for imp, exp in casos:
        res = executar_inferencia(simulador, imp, exp)
        assert 0.0 <= res <= 100.0, f"Falha: Saida {res} fora de [0, 100] para ({imp}, {exp})"


# ------------------------------------------------------------------------------
# 2. TESTES DE MONOTONICIDADE (LOGICA DE NEGOCIO)
# ------------------------------------------------------------------------------
def test_monotonicidade_impacto(simulador):
    """Aumentar o impacto mantendo a explorabilidade deve AUMENTAR ou MANTER a urgencia."""
    exp_fixa = 5.0
    urg_baixa = executar_inferencia(simulador, 20.0, exp_fixa)
    urg_media = executar_inferencia(simulador, 50.0, exp_fixa)
    urg_alta = executar_inferencia(simulador, 90.0, exp_fixa)

    assert urg_baixa <= urg_media <= urg_alta, (
        f"Violacao de Monotonicidade: {urg_baixa:.1f} <= {urg_media:.1f} <= {urg_alta:.1f}"
    )


# ------------------------------------------------------------------------------
# 3. TESTES DE TRATAMENTO DE EXCECAO (ENTRADAS INVALIDAS)
# ------------------------------------------------------------------------------
def test_entradas_invalidas_disparam_erro(simulador):
    """Verifica se o sistema rejeita valores fora do universo do discurso."""
    with pytest.raises(ValueError):
        executar_inferencia(simulador, -10.0, 5.0)  # Impacto negativo

    with pytest.raises(ValueError):
        executar_inferencia(simulador, 150.0, 5.0)  # Impacto > 100

    with pytest.raises(ValueError):
        executar_inferencia(simulador, 50.0, 12.0)  # Explorabilidade > 10


# ------------------------------------------------------------------------------
# 4. TESTE DE DESEMPENHO E LATENCIA (JIT / PERFORMANCE)
# ------------------------------------------------------------------------------
def test_latencia_inferencia(simulador):
    """Garante que 100 inferencias sejam executadas em menos de 1 segundo (SLA < 10ms/op)."""
    inicio = time.time()
    for _ in range(100):
        imp = np.random.uniform(0, 100)
        exp = np.random.uniform(0, 10)
        executar_inferencia(simulador, imp, exp)
    tempo_total = time.time() - inicio

    tempo_medio_ms = (tempo_total / 100) * 1000
    print(f"\nTempo medio por inferencia: {tempo_medio_ms:.3f} ms")

    assert tempo_medio_ms < 10.0, f"Inferencia muito lenta: {tempo_medio_ms:.2f} ms por chamada"

Writing AULA_11/test_fuzzy_engine.py


In [18]:
!touch AULA_11/src/__init__.py

In [19]:
%cd AULA_11
!pytest -v test_fuzzy_engine.py

/content/AULA_11
============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/AULA_11
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.12
collected 0 items / 1 error                                                    

==================================== ERRORS ====================================
____________________ ERROR collecting test_fuzzy_engine.py _____________________
ImportError while importing test module '/content/AULA_11/test_fuzzy_engine.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
test_fuzzy_engine.py:9: in <module>
    from src.engine import construir_motor_inferencia
E   Modu

In [20]:
!ls -la /content/AULA_11/src/

total 16
drwxr-xr-x 3 root root 4096 Oct 29 00:40 .
drwxr-xr-x 5 root root 4096 Oct 29 00:40 ..
-rw-r--r-- 1 root root    0 Oct 29 00:40 __init__.py
-rw-r--r-- 1 root root 1720 Oct 29 00:39 model.py
drwxr-xr-x 2 root root 4096 Oct 29 00:40 __pycache__


In [22]:
!pwd

/content/AULA_11


In [23]:
%%writefile /content/AULA_11/src/engine.py
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from skfuzzy import control as ctrl
from src.model import criar_sistema_fuzzy


def construir_motor_inferencia():
    impacto, explorabilidade, urgencia = criar_sistema_fuzzy()

    regra1 = ctrl.Rule(impacto['baixo'] & explorabilidade['baixa'], urgencia['baixa'])
    regra2 = ctrl.Rule(impacto['baixo'] & explorabilidade['moderada'], urgencia['baixa'])
    regra3 = ctrl.Rule(impacto['baixo'] & explorabilidade['alta'], urgencia['media'])
    regra4 = ctrl.Rule(impacto['medio'] & explorabilidade['baixa'], urgencia['media'])
    regra5 = ctrl.Rule(impacto['medio'] & explorabilidade['moderada'], urgencia['alta'])
    regra6 = ctrl.Rule(impacto['medio'] & explorabilidade['alta'], urgencia['alta'])
    regra7 = ctrl.Rule(impacto['alto'] & explorabilidade['baixa'], urgencia['critica'])
    regra8 = ctrl.Rule(impacto['alto'] & explorabilidade['moderada'], urgencia['critica'])
    regra9 = ctrl.Rule(impacto['alto'] & explorabilidade['alta'], urgencia['critica'])

    sistema_controle = ctrl.ControlSystem([
        regra1, regra2, regra3,
        regra4, regra5, regra6,
        regra7, regra8, regra9
    ])
    return ctrl.ControlSystemSimulation(sistema_controle)


def gerar_superficie_3d(simulador):
    Path("results").mkdir(exist_ok=True)
    x, y = np.meshgrid(np.linspace(0, 100, 20), np.linspace(0, 10, 20))
    z = np.zeros_like(x)

    for i in range(20):
        for j in range(20):
            simulador.input['impacto'] = x[i, j]
            simulador.input['explorabilidade'] = y[i, j]
            simulador.compute()
            z[i, j] = simulador.output['urgencia']

    fig = plt.figure(figsize=(9, 6))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(x, y, z, cmap='viridis', edgecolor='none', alpha=0.9)
    ax.set_xlabel('Impacto no Negocio (%)')
    ax.set_ylabel('Explorabilidade (0-10)')
    ax.set_zlabel('Urgencia Calculada (%)')
    plt.title('Superficie de Decisao Fuzzy - SOC Autonomo')
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5)
    plt.tight_layout()
    plt.savefig('results/superficie_decisao.png')
    plt.close()
    print("Sucesso: Imagem 'results/superficie_decisao.png' gerada com exito!")


def simular_casos_teste(simulador):
    casos = [
        ("Minimo Absoluto", 0.0, 0.0),
        ("Baixo Impacto + Alta Explorabilidade (Transicao)", 10.0, 9.0),
        ("Medio Impacto + Moderado", 50.0, 5.0),
        ("Maximo Absoluto", 100.0, 10.0)
    ]
    print("\n=== SIMULACOES DE TESTE DE INFERENCIA ===")
    for nome, imp, exp in casos:
        simulador.input['impacto'] = imp
        simulador.input['explorabilidade'] = exp
        simulador.compute()
        res = simulador.output['urgencia']
        print(f"[{nome}] -> Imp: {imp}% | Exp: {exp} => Urgencia: {res:.2f}%")


if __name__ == "__main__":
    sim = construir_motor_inferencia()
    simular_casos_teste(sim)
    gerar_superficie_3d(sim)

Writing /content/AULA_11/src/engine.py


In [24]:
!ls -la /content/AULA_11/src/

total 20
drwxr-xr-x 3 root root 4096 Oct 29 00:42 .
drwxr-xr-x 5 root root 4096 Oct 29 00:40 ..
-rw-r--r-- 1 root root 2962 Oct 29 00:42 engine.py
-rw-r--r-- 1 root root    0 Oct 29 00:40 __init__.py
-rw-r--r-- 1 root root 1720 Oct 29 00:39 model.py
drwxr-xr-x 2 root root 4096 Oct 29 00:40 __pycache__


In [25]:
%cd /content/AULA_11
!pytest -v test_fuzzy_engine.py

/content/AULA_11
============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/AULA_11
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.12
collected 4 items                                                              

test_fuzzy_engine.py::test_saida_dentro_do_universo PASSED               [ 25%]
test_fuzzy_engine.py::test_monotonicidade_impacto PASSED                 [ 50%]
test_fuzzy_engine.py::test_entradas_invalidas_disparam_erro PASSED       [ 75%]
test_fuzzy_engine.py::test_latencia_inferencia PASSED                    [100%]

============================== 4 passed in 1.62s ===============================


In [26]:
!pytest -s test_fuzzy_engine.py

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0
rootdir: /content/AULA_11
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.12
collected 4 items                                                              

test_fuzzy_engine.py ...
Tempo medio por inferencia: 3.617 ms
.

============================== 4 passed in 1.61s ===============================


In [27]:
%%writefile /content/AULA_11/test_fuzzy_engine.py
# ==============================================================================
# AULA 11 - SPRINT 3 (AC-3): SUITE DE TESTES AUTOMATIZADOS (PYTEST)
# Objetivo: Validacao de Borda, Monotonicidade e Desempenho do Motor Fuzzy
# VERSAO COM ERRO PROPOSITAL (Execucao 2)
# ==============================================================================

import time
import pytest
import numpy as np
from src.engine import construir_motor_inferencia


@pytest.fixture
def simulador():
    """Fixture que instancia o motor de inferencia compilado para os testes."""
    return construir_motor_inferencia()


def executar_inferencia(simulador, impacto, explorabilidade):
    """Funcao auxiliar para executar inferencia com validacao de entrada."""
    if not (0 <= impacto <= 100):
        raise ValueError(f"Impacto fora da escala [0, 100]: {impacto}")
    if not (0 <= explorabilidade <= 10):
        raise ValueError(f"Explorabilidade fora da escala [0, 10]: {explorabilidade}")

    simulador.input['impacto'] = impacto
    simulador.input['explorabilidade'] = explorabilidade
    simulador.compute()
    return simulador.output['urgencia']


def test_saida_dentro_do_universo(simulador):
    """Garante que a saida esteja sempre no intervalo [0, 100]%."""
    casos = [(0, 0), (50, 5), (100, 10), (30, 8), (80, 2)]
    for imp, exp in casos:
        res = executar_inferencia(simulador, imp, exp)
        assert 0.0 <= res <= 100.0, f"Falha: Saida {res} fora de [0, 100] para ({imp}, {exp})"


def test_monotonicidade_impacto(simulador):
    """Aumentar o impacto mantendo a explorabilidade deve AUMENTAR ou MANTER a urgencia."""
    exp_fixa = 5.0
    urg_baixa = executar_inferencia(simulador, 20.0, exp_fixa)
    urg_media = executar_inferencia(simulador, 50.0, exp_fixa)
    urg_alta = executar_inferencia(simulador, 90.0, exp_fixa)

    # ERRO PROPOSITAL: invertida a logica, exigindo que a urgencia DIMINUA
    # conforme o impacto aumenta - o oposto do comportamento real do sistema.
    assert urg_baixa >= urg_media >= urg_alta, (
        f"Violacao de Monotonicidade: {urg_baixa:.1f} >= {urg_media:.1f} >= {urg_alta:.1f}"
    )


def test_entradas_invalidas_disparam_erro(simulador):
    """Verifica se o sistema rejeita valores fora do universo do discurso."""
    with pytest.raises(ValueError):
        executar_inferencia(simulador, -10.0, 5.0)

    with pytest.raises(ValueError):
        executar_inferencia(simulador, 150.0, 5.0)

    with pytest.raises(ValueError):
        executar_inferencia(simulador, 50.0, 12.0)


def test_latencia_inferencia(simulador):
    """Garante que 100 inferencias sejam executadas em menos de 1 segundo (SLA < 10ms/op)."""
    inicio = time.time()
    for _ in range(100):
        imp = np.random.uniform(0, 100)
        exp = np.random.uniform(0, 10)
        executar_inferencia(simulador, imp, exp)
    tempo_total = time.time() - inicio

    tempo_medio_ms = (tempo_total / 100) * 1000
    print(f"\nTempo medio por inferencia: {tempo_medio_ms:.3f} ms")

    assert tempo_medio_ms < 10.0, f"Inferencia muito lenta: {tempo_medio_ms:.2f} ms por chamada"

Overwriting /content/AULA_11/test_fuzzy_engine.py


In [28]:
!pytest -v test_fuzzy_engine.py

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/AULA_11
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.12
collected 4 items                                                              

test_fuzzy_engine.py::test_saida_dentro_do_universo PASSED               [ 25%]
test_fuzzy_engine.py::test_monotonicidade_impacto FAILED                 [ 50%]
test_fuzzy_engine.py::test_entradas_invalidas_disparam_erro PASSED       [ 75%]
test_fuzzy_engine.py::test_latencia_inferencia PASSED                    [100%]

=================================== FAILURES ===================================
_________________________ test_monotonicidade_impacto __________________________

simulador = <skfuzzy.control.controlsystem.ControlSystemSimulation object at 0x7f53b830f4d0>

    def test_monotonicidade_impacto(simulador):
  

In [29]:
%%writefile /content/AULA_11/test_fuzzy_engine.py
# ==============================================================================
# AULA 11 - SPRINT 3 (AC-3): SUITE DE TESTES AUTOMATIZADOS (PYTEST)
# Objetivo: Validacao de Borda, Monotonicidade e Desempenho do Motor Fuzzy
# ==============================================================================

import time
import pytest
import numpy as np
from src.engine import construir_motor_inferencia


@pytest.fixture
def simulador():
    """Fixture que instancia o motor de inferencia compilado para os testes."""
    return construir_motor_inferencia()


def executar_inferencia(simulador, impacto, explorabilidade):
    """Funcao auxiliar para executar inferencia com validacao de entrada."""
    if not (0 <= impacto <= 100):
        raise ValueError(f"Impacto fora da escala [0, 100]: {impacto}")
    if not (0 <= explorabilidade <= 10):
        raise ValueError(f"Explorabilidade fora da escala [0, 10]: {explorabilidade}")

    simulador.input['impacto'] = impacto
    simulador.input['explorabilidade'] = explorabilidade
    simulador.compute()
    return simulador.output['urgencia']


def test_saida_dentro_do_universo(simulador):
    """Garante que a saida esteja sempre no intervalo [0, 100]%."""
    casos = [(0, 0), (50, 5), (100, 10), (30, 8), (80, 2)]
    for imp, exp in casos:
        res = executar_inferencia(simulador, imp, exp)
        assert 0.0 <= res <= 100.0, f"Falha: Saida {res} fora de [0, 100] para ({imp}, {exp})"


def test_monotonicidade_impacto(simulador):
    """Aumentar o impacto mantendo a explorabilidade deve AUMENTAR ou MANTER a urgencia."""
    exp_fixa = 5.0
    urg_baixa = executar_inferencia(simulador, 20.0, exp_fixa)
    urg_media = executar_inferencia(simulador, 50.0, exp_fixa)
    urg_alta = executar_inferencia(simulador, 90.0, exp_fixa)

    assert urg_baixa <= urg_media <= urg_alta, (
        f"Violacao de Monotonicidade: {urg_baixa:.1f} <= {urg_media:.1f} <= {urg_alta:.1f}"
    )


def test_entradas_invalidas_disparam_erro(simulador):
    """Verifica se o sistema rejeita valores fora do universo do discurso."""
    with pytest.raises(ValueError):
        executar_inferencia(simulador, -10.0, 5.0)

    with pytest.raises(ValueError):
        executar_inferencia(simulador, 150.0, 5.0)

    with pytest.raises(ValueError):
        executar_inferencia(simulador, 50.0, 12.0)


def test_latencia_inferencia(simulador):
    """Garante que 100 inferencias sejam executadas em menos de 1 segundo (SLA < 10ms/op)."""
    inicio = time.time()
    for _ in range(100):
        imp = np.random.uniform(0, 100)
        exp = np.random.uniform(0, 10)
        executar_inferencia(simulador, imp, exp)
    tempo_total = time.time() - inicio

    tempo_medio_ms = (tempo_total / 100) * 1000
    print(f"\nTempo medio por inferencia: {tempo_medio_ms:.3f} ms")

    assert tempo_medio_ms < 10.0, f"Inferencia muito lenta: {tempo_medio_ms:.2f} ms por chamada"

Overwriting /content/AULA_11/test_fuzzy_engine.py


In [30]:
!pytest -v test_fuzzy_engine.py

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/AULA_11
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.12
collected 4 items                                                              

test_fuzzy_engine.py::test_saida_dentro_do_universo PASSED               [ 25%]
test_fuzzy_engine.py::test_monotonicidade_impacto PASSED                 [ 50%]
test_fuzzy_engine.py::test_entradas_invalidas_disparam_erro PASSED       [ 75%]
test_fuzzy_engine.py::test_latencia_inferencia PASSED                    [100%]

============================== 4 passed in 1.57s ===============================
